In [2]:
import pandas as pd
import requests
import geopandas as gpd


# Gather Crop Yield(corn,soybean,wheat) History for Tennessee from 2010 to 2024. 

In [2]:
# create private API key
import json
with open('keys.json') as fi:
    credentials = json.load(fi)
usda_crop_api_key = credentials['usda_crop_api_key']
noaa_weather_api_key = credentials['noaa_weather_api_key']


In [3]:
# Base URL for USDA NASS Quick Stats API 
crop_base_url = 'https://quickstats.nass.usda.gov/api/api_GET/'

crop_records = []

field_crops = ['CORN','SOYBEANS', 'WHEAT']

for field_crop in field_crops:
    for year in range(2000, 2025):  # Excludes 2025
        
# Define Query Parameters for filtering the data set   
# Get the keys from https://quickstats.nass.usda.gov/api from 'usage page'
        
        params = {
            'key': usda_crop_api_key,
            'commodity_desc': field_crop,
            'year': str(year),
            'state_name': 'TENNESSEE',
            'statisticcat_desc': 'YIELD', 
            'unit_desc': 'BU / ACRE'
        }
        
      # Send Get Request to USDA API with Parameters
        crop_response = requests.get(crop_base_url, params=params)
      # Convert API Response in to Python dictonary
        crop_data = crop_response.json()
        #data.keys()   # get key dictionary
        if "data" in crop_data:
            crop_records.extend(crop_data["data"])


In [4]:
print(type(crop_records))
print(len(crop_records))

<class 'list'>
5384


In [5]:
#Convert crop yield from list to Data frame
crop_df = pd.DataFrame(crop_records)

In [6]:
# Export Raw Crop Data frame to csv file
crop_df.to_csv('../data/Raw_Crop_df', index=False)

In [7]:
crop_df.head()

,class_desc,country_name,unit_desc,commodity_desc,group_desc,statisticcat_desc,prodn_practice_desc,freq_desc,county_ansi,CV (%),...,location_desc,watershed_desc,Value,sector_desc,domain_desc,agg_level_desc,end_code,asd_code,util_practice_desc,load_time
0,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, DELTA",,131,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,10,GRAIN,2012-01-01 00:00:00.000
1,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, WEST TENNESSEE",,109,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,20,GRAIN,2012-01-01 00:00:00.000
2,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, WESTERN RIM",,106,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,30,GRAIN,2012-01-01 00:00:00.000
3,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, CENTRAL BASIN",,102,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,40,GRAIN,2012-01-01 00:00:00.000
4,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, CUMBERLAND PLATEAU",,125,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,50,GRAIN,2012-01-01 00:00:00.000


In [8]:
crop_df[['commodity_desc','year','state_name','statisticcat_desc','unit_desc','Value','county_name']]

,commodity_desc,year,state_name,statisticcat_desc,unit_desc,Value,county_name
0,CORN,2000,TENNESSEE,YIELD,BU / ACRE,131,
1,CORN,2000,TENNESSEE,YIELD,BU / ACRE,109,
2,CORN,2000,TENNESSEE,YIELD,BU / ACRE,106,
3,CORN,2000,TENNESSEE,YIELD,BU / ACRE,102,
4,CORN,2000,TENNESSEE,YIELD,BU / ACRE,125,
...,...,...,...,...,...,...,...
5379,WHEAT,2024,TENNESSEE,YIELD,BU / ACRE,75,
5380,WHEAT,2024,TENNESSEE,YIELD,BU / ACRE,76,
5381,WHEAT,2024,TENNESSEE,YIELD,BU / ACRE,76,
5382,WHEAT,2024,TENNESSEE,YIELD,BU / ACRE,76,


In [9]:
crop_df["county_name"].unique()

array(['', 'DYER', 'LAKE', 'LAUDERDALE', 'OBION', 'SHELBY', 'TIPTON',
       'BENTON', 'CARROLL', 'CHESTER', 'CROCKETT', 'DECATUR', 'FAYETTE',
       'GIBSON', 'HARDEMAN', 'HARDIN', 'HAYWOOD', 'HENDERSON', 'HENRY',
       'MCNAIRY', 'MADISON', 'WEAKLEY', 'OTHER (COMBINED) COUNTIES',
       'CHEATHAM', 'DICKSON', 'HICKMAN', 'HUMPHREYS', 'LAWRENCE',
       'MONTGOMERY', 'PERRY', 'ROBERTSON', 'STEWART', 'WAYNE', 'BEDFORD',
       'CANNON', 'DAVIDSON', 'DE KALB', 'GILES', 'LINCOLN', 'MACON',
       'MARSHALL', 'MAURY', 'MOORE', 'RUTHERFORD', 'SMITH', 'SUMNER',
       'TROUSDALE', 'WILLIAMSON', 'WILSON', 'BLEDSOE', 'COFFEE',
       'CUMBERLAND', 'FENTRESS', 'FRANKLIN', 'GRUNDY', 'MARION',
       'OVERTON', 'SEQUATCHIE', 'WARREN', 'WHITE', 'BLOUNT', 'BRADLEY',
       'COCKE', 'GREENE', 'HAMBLEN', 'JEFFERSON', 'MCMINN', 'MEIGS',
       'MONROE', 'POLK', 'RHEA', 'WASHINGTON', 'PUTNAM', 'CLAIBORNE',
       'GRAINGER', 'HAWKINS', 'JOHNSON', 'SULLIVAN', 'LOUDON', 'HAMILTON',
       'CARTER', 'CLA

In [10]:
(crop_df["county_name"] == "").sum()

np.int64(887)

In [11]:
crop_df[
    ~crop_df["county_name"].isin(
        ["OTHER COUNTIES", "OTHER (COMBINED) COUNTIES"]
    )
]

,class_desc,country_name,unit_desc,commodity_desc,group_desc,statisticcat_desc,prodn_practice_desc,freq_desc,county_ansi,CV (%),...,location_desc,watershed_desc,Value,sector_desc,domain_desc,agg_level_desc,end_code,asd_code,util_practice_desc,load_time
0,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, DELTA",,131,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,10,GRAIN,2012-01-01 00:00:00.000
1,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, WEST TENNESSEE",,109,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,20,GRAIN,2012-01-01 00:00:00.000
2,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, WESTERN RIM",,106,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,30,GRAIN,2012-01-01 00:00:00.000
3,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, CENTRAL BASIN",,102,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,40,GRAIN,2012-01-01 00:00:00.000
4,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,"TENNESSEE, CUMBERLAND PLATEAU",,125,CROPS,TOTAL,AGRICULTURAL DISTRICT,00,50,GRAIN,2012-01-01 00:00:00.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5379,WINTER,UNITED STATES,BU / ACRE,WHEAT,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,TENNESSEE,,75,CROPS,TOTAL,STATE,00,,ALL UTILIZATION PRACTICES,2025-09-30 12:00:00.000
5380,WINTER,UNITED STATES,BU / ACRE,WHEAT,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,TENNESSEE,,76,CROPS,TOTAL,STATE,00,,ALL UTILIZATION PRACTICES,2024-08-12 12:00:00.000
5381,WINTER,UNITED STATES,BU / ACRE,WHEAT,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,TENNESSEE,,76,CROPS,TOTAL,STATE,00,,ALL UTILIZATION PRACTICES,2024-07-12 12:00:00.000
5382,WINTER,UNITED STATES,BU / ACRE,WHEAT,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,,,...,TENNESSEE,,76,CROPS,TOTAL,STATE,00,,ALL UTILIZATION PRACTICES,2024-06-12 12:00:00.000


# Later Clean Crops data frame 

# Gather weather History from 2010 to 2024

Get Api from this link https://www.ncei.noaa.gov/cdo-web/webservices/v2#data for the weather history

In [12]:
# Call Stations end point to get stationsid to find the Countiesafter merging it weather data
# stations_base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

# headers = {
#     "token": noaa_weather_api_key
# }

# params = {
#     "datasetid": "GSOM",
#     "locationid": "FIPS:47",
#     "startdate": "2010-01-01",
#     "enddate": "2024-12-31",
#     "limit": 1000
# }

# req = requests.get(stations_base_url,headers=headers,params=params)
# stations = req.json()
# stations.keys()
# stations_repo = stations['results']

In [13]:

#stations = pd.DataFrame(stations_repo)


In [14]:
# Call Stations end point to get stationsid to find the Countiesafter merging it weather data
stations_base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

# Authentication token
headers = {
    "token": noaa_weather_api_key
}

# Query Parameters to filter stations
params = {
    "datasetid": "GSOM",         # Monthly Summary Data
    "locationid": "FIPS:47",     # TN state Fips code 47
    "startdate": "2000-01-01",   # Start date filter for station availabilty
    "enddate": "2024-12-31",     # End date filter
    "limit": 1000,               # Max Records per one API Request
    "offset": 1                  # pagination start index
}

# List to store all station records
all_stations = []

# Loop to handle pagination
while True:

    # Make API request to NOAA stations endpoint
    response = requests.get(stations_base_url, headers=headers, params=params)

    # Stop if API call fails
    if response.status_code != 200:
        print("Request failed:", response.text)
        break

    # Convert JSON response to Python dictionary
    data = response.json()

    # Extract station records from response
    results = data.get("results", [])

    # stop if no more data
    if not results:
        break

    # Add current page results to all_stations list
    all_stations.extend(results)

    # if last page less than limit, stop
    if len(results) < params["limit"]:
        break

    #Move to next page using offset pagination
    params["offset"] += params["limit"]

# Convert all stations list to Dataframe
stations_df = pd.DataFrame(all_stations)

In [15]:
stations_df.head()

,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude
0,282.5,2008-01-01,2015-06-01,35.011389,"ARDMORE 1.3 NE, TN US",0.3224,GHCND:US1CHARM185,METERS,-86.839167
1,270.1,2007-05-01,2026-05-01,36.019079,"OAK RIDGE 5.7 NE, TN US",0.8864,GHCND:US1TNAN0003,METERS,-84.221136
2,360.0,2008-06-01,2009-06-01,36.008300,"OAK RIDGE 3.3 NNW, TN US",0.7706,GHCND:US1TNAN0004,METERS,-84.311200
3,274.3,2007-05-01,2008-10-01,36.031200,"CLINTON 4.3 SSW, TN US",0.6088,GHCND:US1TNAN0006,METERS,-84.150000
4,353.0,2007-10-01,2026-05-01,36.199700,"NORRIS 0.6 NW, TN US",0.9063,GHCND:US1TNAN0008,METERS,-84.076500


In [16]:
stations_df.shape

(1351, 9)

In [17]:
# Export Raw stations Data frame to csv file
stations_df.to_csv('../data/Raw_stations_df', index=False)

In [18]:
stations_df[stations_df['id']=='GHCND:USC00155694']

,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude
1143,150.0,1926-04-01,2026-05-01,36.61214,"MURRAY, KY US",0.9559,GHCND:USC00155694,METERS,-88.30116


spatialjoins, find the nearest station for there are no counties

Get Api from this link https://www.ncei.noaa.gov/cdo-web/webservices/v2#data for the weather history

In [19]:
# weather_base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"

# headers = {
#     "token": noaa_weather_api_key
# }

# params = {
                
#             "datasetid": "GSOM",
#             "locationid": "FIPS:47",
#             "startdate": "2010-01-01",
#             "enddate": "2017-12-31",
#             "datatypeid": "TMIN",
#             "limit": 1000
#         }
# weather_response = requests.get(weather_base_url,headers=headers, params=params)
# weather = weather_response.json()
# weather.keys()
# weather_response

In [20]:
#weather_response.text

In [21]:
# weather_repo = weather['results']
# pd.DataFrame(weather_repo)

In [22]:
# Base URL for NCEI NOAA WEATHER API 

weather_base_url = 'https://www.ncei.noaa.gov/cdo-web/api/v2/data'

headers = {
    "token": noaa_weather_api_key
}

all_weather = []

dataTypes = ['PRCP','TMAX', 'TMIN']

for dataType in dataTypes:
    
    for year in range(2000, 2025):
        offset = 1  # pagination start
        
        while True:
        # Define Query Parameters for filtering the data set  
        # Get the keys from https://www.ncei.noaa.gov/cdo-web/webservices/v2#data 
            params = {
                "datasetid": "GSOM",
                "locationid": "FIPS:47",
                "startdate": f"{year}-01-01",
                "enddate": f"{year}-12-31",
                "datatypeid": dataType,
                "units" : "standard",
                "limit": 1000,
                "offset": offset
            }
                
              # Send Get Request to ncei NOAA  API with Parameters
            weather_response = requests.get(weather_base_url, headers=headers, params=params, timeout=60)
              # Convert API Response in to Python dictonary
            weather_data = weather_response.json()
            if weather_response.status_code != 200:
                print(f"Failed: {dataType} {year}")
                break

            results = weather_data.get("results",[])

            if len(results) == 0:
                break

            all_weather.extend(results)

            if len(results) < 1000:
                break

            offset += 1000

            #weather_data.keys()
            #if "results" in weather_data:
                #all_weather.extend(weather_data["results"])
            #offset += 1000
weather_df = pd.DataFrame(all_weather)

In [23]:
#Export Raw weather Data frame to csv file
weather_df.to_csv('../data/Raw_weather_df', index=False)

In [24]:
weather_df.head()

,date,datatype,station,attributes,value
0,2000-01-01T00:00:00,PRCP,GHCND:US1TNWL0002,",,,N",2.26
1,2000-01-01T00:00:00,PRCP,GHCND:USC00155694,",,,0",3.55
2,2000-01-01T00:00:00,PRCP,GHCND:USC00400081,",,,0",4.18
3,2000-01-01T00:00:00,PRCP,GHCND:USC00400137,",,,0",2.31
4,2000-01-01T00:00:00,PRCP,GHCND:USC00400284,",,,0",5.16


In [25]:
weather_df.shape

(185293, 5)

## Get the County shape file data (Tiger/Line shape files) https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/

In [3]:
# Read County Zip file
counties = gpd.read_file('../data/tl_2023_us_county.zip')
counties.head()

,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,31,039,00835841,31039,0500000US31039,Cuming,Cuming County,06,H1,G4020,None,None,None,A,1477563029,10772508,+41.9158651,-096.7885168,"POLYGON ((-96.55516 41.91587, -96.55515 41.914..."
1,53,069,01513275,53069,0500000US53069,Wahkiakum,Wahkiakum County,06,H1,G4020,None,None,None,A,680980771,61564427,+46.2946377,-123.4244583,"POLYGON ((-123.72755 46.2645, -123.72756 46.26..."
2,35,011,00933054,35011,0500000US35011,De Baca,De Baca County,06,H1,G4020,None,None,None,A,6016818946,29090018,+34.3592729,-104.3686961,"POLYGON ((-104.89337 34.08894, -104.89337 34.0..."
3,31,109,00835876,31109,0500000US31109,Lancaster,Lancaster County,06,H1,G4020,339,30700,None,A,2169269688,22850324,+40.7835474,-096.6886584,"POLYGON ((-96.68493 40.5233, -96.69219 40.5231..."
4,31,129,00835886,31129,0500000US31129,Nuckolls,Nuckolls County,06,H1,G4020,None,None,None,A,1489645187,1718484,+40.1764918,-098.0468422,"POLYGON ((-98.2737 40.1184, -98.27374 40.1224,..."


In [4]:
# Get only TN Counties 
tn_counties = counties[counties["STATEFP"] == "47"]

In [5]:
tn_counties.head()

,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
29,47,065,01639749,47065,0500000US47065,Hamilton,Hamilton County,06,H1,G4020,174,16860,None,A,1404197535,86784458,+35.1634720,-085.2018432,"POLYGON ((-85.17523 34.98598, -85.17663 34.986..."
64,47,115,01639770,47115,0500000US47115,Marion,Marion County,06,H1,G4020,174,16860,None,A,1290472798,36484703,+35.1334215,-085.6183990,"POLYGON ((-85.86396 34.99316, -85.86395 34.993..."
67,47,185,01639800,47185,0500000US47185,White,White County,06,H1,G4020,None,18260,None,A,975592109,7113368,+35.9270486,-085.4557854,"MULTIPOLYGON (((-85.6446 36.01505, -85.64392 3..."
127,47,129,01639778,47129,0500000US47129,Morgan,Morgan County,06,H1,G4020,315,28940,None,A,1352439680,823018,+36.1386970,-084.6392616,"POLYGON ((-84.45834 36.18654, -84.45834 36.186..."
180,47,013,01639728,47013,0500000US47013,Campbell,Campbell County,06,H1,G4020,315,28940,None,A,1243685306,46425147,+36.4015922,-084.1592495,"POLYGON ((-84.09196 36.22261, -84.09202 36.222..."


In [6]:
# Export tn_counties to csv file
tn_counties.to_csv('../data/Raw_tn_counties_df', index=False)

In [7]:
# Take only county name and polygons
tn_counties = tn_counties[["NAME", "geometry"]]
tn_counties = tn_counties.rename(columns={"NAME": "county"})

In [8]:
tn_counties.head()

,county,geometry
29,Hamilton,"POLYGON ((-85.17523 34.98598, -85.17663 34.986..."
64,Marion,"POLYGON ((-85.86396 34.99316, -85.86395 34.993..."
67,White,"MULTIPOLYGON (((-85.6446 36.01505, -85.64392 3..."
127,Morgan,"POLYGON ((-84.45834 36.18654, -84.45834 36.186..."
180,Campbell,"POLYGON ((-84.09196 36.22261, -84.09202 36.222..."


### Convert Pandas Stations df into geospatial df to do spatial Operations

In [32]:
#Convert pandas stations df to geospatial df
stations_gdf = gpd.GeoDataFrame(
    stations_df,
    #Create Geometry Column using longitude and latitude
    geometry=gpd.points_from_xy(stations_df.longitude, stations_df.latitude),
    # Create Coordinate system of input coordinates
    crs="EPSG:4326"
)

In [33]:
stations_gdf.head()

,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude,geometry
0,282.5,2008-01-01,2015-06-01,35.011389,"ARDMORE 1.3 NE, TN US",0.3224,GHCND:US1CHARM185,METERS,-86.839167,POINT (-86.83917 35.01139)
1,270.1,2007-05-01,2026-05-01,36.019079,"OAK RIDGE 5.7 NE, TN US",0.8864,GHCND:US1TNAN0003,METERS,-84.221136,POINT (-84.22114 36.01908)
2,360.0,2008-06-01,2009-06-01,36.008300,"OAK RIDGE 3.3 NNW, TN US",0.7706,GHCND:US1TNAN0004,METERS,-84.311200,POINT (-84.3112 36.0083)
3,274.3,2007-05-01,2008-10-01,36.031200,"CLINTON 4.3 SSW, TN US",0.6088,GHCND:US1TNAN0006,METERS,-84.150000,POINT (-84.15 36.0312)
4,353.0,2007-10-01,2026-05-01,36.199700,"NORRIS 0.6 NW, TN US",0.9063,GHCND:US1TNAN0008,METERS,-84.076500,POINT (-84.0765 36.1997)


In [34]:
# Change the coordinate system of stations to match with count shape files data (if needed)
stations_gdf = stations_gdf.to_crs(tn_counties.crs)

In [35]:
print(stations_gdf.crs)
print(tn_counties.crs)

EPSG:4269
EPSG:4269


In [36]:
# Join Weather Stations to Counties
stations_county = gpd.sjoin(
    stations_gdf,
    tn_counties,
    how="left",
    predicate="within" #Match if station lies inside county boundary
)

In [37]:
stations_county[stations_county["county"].isna()]

,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude,geometry,index_right,county
1143,150.0,1926-04-01,2026-05-01,36.61214,"MURRAY, KY US",0.9559,GHCND:USC00155694,METERS,-88.30116,POINT (-88.30116 36.61214),NaN,NaN


In [38]:
# Find the nearest county based on coordinates by using sjoin_nearest for missed county
inside_stations = stations_county.copy()

In [39]:

missing = inside_stations[inside_stations["county"].isna()]


In [40]:
# join tn_counties and missing data
nearest = gpd.sjoin_nearest(
    #Remove geopandas join helper column, do not need it
    missing.drop(columns=["index_right"]),
    tn_counties,
    how="left",
    distance_col="dist_km"
)

/opt/anaconda3/lib/python3.13/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


In [41]:
# Concatenate actual county stations and the rows which has neares county
stations_with_county = pd.concat(
    [inside_stations[~inside_stations["county"].isna()], nearest],
    ignore_index=True
)


In [42]:
stations_with_county[stations_with_county["county"].isna()]

,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude,geometry,index_right,county,county_left,county_right,dist_km
1350,150.0,1926-04-01,2026-05-01,36.61214,"MURRAY, KY US",0.9559,GHCND:USC00155694,METERS,-88.30116,POINT (-88.30116 36.61214),1564.0,NaN,NaN,Henry,0.111739


In [43]:
stations_with_county["final_county"] = stations_with_county["county"].fillna(stations_with_county["county_right"])

In [44]:
# Drop unnecessary extra county columns after joining
stations_with_county = stations_with_county.drop(columns=[
    "county", "county_right", "index_right"
])

In [45]:
# Rename
stations_with_county= stations_with_county.rename(columns={"final_county": "county"})

In [46]:
stations_with_county.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1351 entries, 0 to 1350
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   elevation      1350 non-null   float64 
 1   mindate        1351 non-null   object  
 2   maxdate        1351 non-null   object  
 3   latitude       1351 non-null   float64 
 4   name           1351 non-null   object  
 5   datacoverage   1351 non-null   float64 
 6   id             1351 non-null   object  
 7   elevationUnit  1350 non-null   object  
 8   longitude      1351 non-null   float64 
 9   geometry       1351 non-null   geometry
 10  county_left    0 non-null      object  
 11  dist_km        1 non-null      float64 
 12  county         1351 non-null   object  
dtypes: float64(5), geometry(1), object(7)
memory usage: 137.3+ KB


In [47]:
# Check if there are any missing values of county column, if any use gpd.sjpoin_nearest assign weather station to a nearest county
stations_with_county["county"].isna().sum()

np.int64(0)

In [48]:
#Export Raw weather Data frame to csv file
stations_with_county.to_csv('../data/stations_with_county_df', index=False)

#### Convert datatypes of date column in weather df

In [49]:
weather_df["date"] = pd.to_datetime(weather_df["date"])
weather_df["year"] = weather_df["date"].dt.year

In [50]:
weather_df.head()

,date,datatype,station,attributes,value,year
0,2000-01-01,PRCP,GHCND:US1TNWL0002,",,,N",2.26,2000
1,2000-01-01,PRCP,GHCND:USC00155694,",,,0",3.55,2000
2,2000-01-01,PRCP,GHCND:USC00400081,",,,0",4.18,2000
3,2000-01-01,PRCP,GHCND:USC00400137,",,,0",2.31,2000
4,2000-01-01,PRCP,GHCND:USC00400284,",,,0",5.16,2000


In [51]:
# Reshape weather df from long format to wide format
weather_pivot = weather_df.pivot_table(
    #Group records by station and year
    index=["station", "year"],
    # Create seperate columns for each weather datatype variable
    columns="datatype",
    values="value",
    # If multiple values take avg
    aggfunc="mean"
    #Convert grouped index back to normal columns
).reset_index()

In [52]:
weather_pivot.head()

datatype,station,year,PRCP,TMAX,TMIN
0,GHCND:US1CHARM185,2008,3.830000,NaN,NaN
1,GHCND:US1CHARM185,2009,4.990000,NaN,NaN
2,GHCND:US1CHARM185,2014,4.870000,NaN,NaN
3,GHCND:US1CHARM185,2015,4.551667,NaN,NaN
4,GHCND:US1TNAN0003,2007,2.831250,NaN,NaN


#### Now Merge pivoted Weather df to stations with county df

In [53]:
# Merge pivoted Weather df to stations with county df
weather_station_county = weather_pivot.merge(
    stations_with_county[["id", "county"]],
    left_on="station",
    right_on="id",
    how="left"
)

In [54]:
weather_station_county

,station,year,PRCP,TMAX,TMIN,id,county
0,GHCND:US1CHARM185,2008,3.830000,NaN,NaN,GHCND:US1CHARM185,Giles
1,GHCND:US1CHARM185,2009,4.990000,NaN,NaN,GHCND:US1CHARM185,Giles
2,GHCND:US1CHARM185,2014,4.870000,NaN,NaN,GHCND:US1CHARM185,Giles
3,GHCND:US1CHARM185,2015,4.551667,NaN,NaN,GHCND:US1CHARM185,Giles
4,GHCND:US1TNAN0003,2007,2.831250,NaN,NaN,GHCND:US1TNAN0003,Anderson
...,...,...,...,...,...,...,...
10885,GHCND:USW00063855,2020,5.874167,65.183333,45.691667,GHCND:USW00063855,Cumberland
10886,GHCND:USW00063855,2021,4.937500,65.033333,45.066667,GHCND:USW00063855,Cumberland
10887,GHCND:USW00063855,2022,5.442500,64.933333,43.875000,GHCND:USW00063855,Cumberland
10888,GHCND:USW00063855,2023,4.472500,66.008333,45.191667,GHCND:USW00063855,Cumberland


In [55]:
weather_station_county[weather_station_county['county'].isna()]

,station,year,PRCP,TMAX,TMIN,id,county
254,GHCND:US1TNBF0027,2008,2.21,NaN,NaN,NaN,NaN


One station GHCND:USC00155694 contains 14 rows and is in weather df but not in the stations df so it does not have the coordinates.
There is no chance to find the nearest countu to fill the missing counties. So dropping those missing county rows.

In [56]:
weather_station_county = weather_station_county.dropna(subset=["county"])

In [57]:
weather_station_county[weather_station_county['county'].isna()]

,station,year,PRCP,TMAX,TMIN,id,county


In [58]:
#Export  weather station county Data frame to csv file
weather_station_county.to_csv('../data/weather_station_county_df', index=False)

In [59]:
weather_station_county[weather_station_county['county'].isna()]

,station,year,PRCP,TMAX,TMIN,id,county


In [60]:
weather_station_county.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10889 entries, 0 to 10889
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   station  10889 non-null  object 
 1   year     10889 non-null  int32  
 2   PRCP     10506 non-null  float64
 3   TMAX     3839 non-null   float64
 4   TMIN     3832 non-null   float64
 5   id       10889 non-null  object 
 6   county   10889 non-null  object 
dtypes: float64(3), int32(1), object(3)
memory usage: 638.0+ KB


In [61]:
# Aggregate Station level weather data into county year average weather features(PRCP, TMAX, TMIN)
county_weather = weather_station_county.groupby(
    ["county", "year"]
).agg({
    "PRCP": "mean",
    "TMAX": "mean",
    "TMIN": "mean"
}).reset_index()

In [62]:
county_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2315 entries, 0 to 2314
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   county  2315 non-null   object 
 1   year    2315 non-null   int32  
 2   PRCP    2307 non-null   float64
 3   TMAX    2155 non-null   float64
 4   TMIN    2152 non-null   float64
dtypes: float64(3), int32(1), object(1)
memory usage: 81.5+ KB


In [63]:
#Export  county weather Data frame to csv file
county_weather.to_csv('../data/final_county_weather_df', index=False)

In [64]:
county_weather.shape

(2315, 5)

# Gather Soil Data

#### Down load data set from this soil Survey Geographic database https://www.nrcs.usda.gov/resources/data-and-reports/gridded-soil-survey-geographic-gssurgo-database
click on State Databases - soils | Powered by Box

In [9]:
# Read SSURGO soil map unit polygons for Tennessee.
# Each row represents a soil unit boundary with geometry and MUKEY identifier.
soil_keys = gpd.read_file("../data/gSSURGO_TN.gdb", layer="MUPOLYGON") #MULTIPOLYGON - a single geometric object that represents a collection of multiple, non-overlapping polygons treated as one feature

/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [10]:
soil_keys.head()

,AREASYMBOL,SPATIALVER,MUSYM,MUKEY,Shape_Length,Shape_Area,geometry
0,TN091,5.0,CaF,526516,5236.926602,534416.62,"MULTIPOLYGON (((1259534.4 1600055.8, 1259510.8..."
1,TN091,5.0,CcE,526518,2847.382752,223007.82,"MULTIPOLYGON (((1244721.2 1577069.9, 1244679.2..."
2,TN091,5.0,DtE,526529,2299.663378,79289.81,"MULTIPOLYGON (((1259721.8 1591012.2, 1259693 1..."
3,TN091,5.0,KeC,526541,1132.074101,77883.95,"MULTIPOLYGON (((1258520.1 1590386.1, 1258526.7..."
4,TN091,5.0,CaE,526515,964.949459,48297.57,"MULTIPOLYGON (((1242604.1 1578393.6, 1242606.3..."


In [11]:
# Load soil attribute table (Muaggatt) from SSURGO geodatabase
# This table contains soil properties (e.g., slope, drainage, water capacity)
# Each record is identified by MUKEY, which links to soil polygon geometries
muaggatt_df =gpd.read_file("../data/gSSURGO_TN.gdb", layer="Muaggatt")

In [12]:
muaggatt_df.head()

,musym,muname,mustatus,slopegraddcp,slopegradwta,brockdepmin,wtdepannmin,wtdepaprjunmin,flodfreqdcd,flodfreqmax,...,engsldcp,englrsdcd,engcmssdcd,engcmssmp,urbrecptdcd,urbrecptwta,forpehrtdcp,hydclprs,awmmfpwwta,mukey
0,AcF,"Ashe-Cleveland-Rock outcrop complex, 50 to 95 ...",None,73.0,73.0,0.0,NaN,NaN,None,None,...,Very limited,Very limited,Fair,Not rated,Very limited,1.00,Erosion hazard very severe,0,1.000,526504
1,AsE,"Ashe gravelly fine sandy loam, 12 to 25 percen...",None,19.0,19.0,81.0,NaN,NaN,None,None,...,Very limited,Very limited,Fair,Fair,Somewhat limited,0.32,Erosion hazard moderate,0,1.000,526505
2,AsF,"Ashe gravelly fine sandy loam, 25 to 65 percen...",None,45.0,45.0,81.0,NaN,NaN,None,None,...,Very limited,Very limited,Fair,Fair,Very limited,1.00,Erosion hazard severe,0,1.000,526506
3,BeC,"Bledsoe silt loam, 5 to 12 percent slopes",None,9.0,9.0,NaN,NaN,NaN,None,None,...,Very limited,Very limited,Poor,Poor,Very limited,1.00,Erosion hazard severe,0,0.412,526507
4,BeD,"Bledsoe silt loam, 12 to 20 percent slopes",None,16.0,16.0,NaN,NaN,NaN,None,None,...,Very limited,Very limited,Poor,Poor,Very limited,1.00,Erosion hazard very severe,0,1.000,526508


In [13]:
muaggatt_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6870 entries, 0 to 6869
Data columns (total 40 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   musym           6870 non-null   object 
 1   muname          6870 non-null   object 
 2   mustatus        0 non-null      object 
 3   slopegraddcp    6686 non-null   float32
 4   slopegradwta    6697 non-null   float32
 5   brockdepmin     2021 non-null   float64
 6   wtdepannmin     1890 non-null   float64
 7   wtdepaprjunmin  1322 non-null   float64
 8   flodfreqdcd     6608 non-null   object 
 9   flodfreqmax     6638 non-null   object 
 10  pondfreqprs     6870 non-null   object 
 11  aws025wta       6564 non-null   float32
 12  aws050wta       6564 non-null   float32
 13  aws0100wta      6564 non-null   float32
 14  aws0150wta      6564 non-null   float32
 15  drclassdcd      6487 non-null   object 
 16  drclasswettest  6567 non-null   object 
 17  hydgrpdcd       6490 non-null   o

In [14]:
muaggatt_df['muname']

0       Ashe-Cleveland-Rock outcrop complex, 50 to 95 ...
1       Ashe gravelly fine sandy loam, 12 to 25 percen...
2       Ashe gravelly fine sandy loam, 25 to 65 percen...
3               Bledsoe silt loam, 5 to 12 percent slopes
4              Bledsoe silt loam, 12 to 20 percent slopes
                              ...                        
6865                                                Water
6866    Waverly silt loam, 0 to 2 percent slopes, occa...
6867                                Mines and Gravel Pits
6868    Adler silt loam, 0 to 2 percent slopes, occasi...
6869    Roellen silty clay loam, 0 to 1 percent slopes...
Name: muname, Length: 6870, dtype: object

Merge soil spatial polygons with soil attribute table.
soil_keys contains geometry (soil map unit boundaries).
muaggatt_df contains soil properties (slope, drainage, water capacity, etc.).
We join using MUKEY (from spatial data) and mukey (from attributes table).
This attaches soil characteristics to each soil polygon.

In [15]:
soil_gdf = soil_keys.merge(muaggatt_df,left_on="MUKEY",right_on="mukey",how="left")

In [16]:
soil_gdf.head()

,AREASYMBOL,SPATIALVER,MUSYM,MUKEY,Shape_Length,Shape_Area,geometry,musym,muname,mustatus,...,engsldcp,englrsdcd,engcmssdcd,engcmssmp,urbrecptdcd,urbrecptwta,forpehrtdcp,hydclprs,awmmfpwwta,mukey
0,TN091,5.0,CaF,526516,5236.926602,534416.62,"MULTIPOLYGON (((1259534.4 1600055.8, 1259510.8...",CaF,"Calvin channery silt loam, 35 to 50 percent sl...",None,...,Very limited,Very limited,Poor,Poor,Very limited,1.0,Erosion hazard severe,0,1.000,526516
1,TN091,5.0,CcE,526518,2847.382752,223007.82,"MULTIPOLYGON (((1244721.2 1577069.9, 1244679.2...",CcE,"Cataska channery silt loam, 20 to 35 percent s...",None,...,Very limited,Very limited,Poor,Fair,Very limited,1.0,Erosion hazard moderate,0,1.000,526518
2,TN091,5.0,DtE,526529,2299.663378,79289.81,"MULTIPOLYGON (((1259721.8 1591012.2, 1259693 1...",DtE,"Ditney sandy loam, 20 to 35 percent slopes",None,...,Very limited,Very limited,Fair,Fair,Very limited,1.0,Erosion hazard moderate,0,1.000,526529
3,TN091,5.0,KeC,526541,1132.074101,77883.95,"MULTIPOLYGON (((1258520.1 1590386.1, 1258526.7...",KeC,"Keener loam, 5 to 12 percent slopes",None,...,Very limited,Very limited,Poor,Poor,Very limited,1.0,Erosion hazard severe,0,0.622,526541
4,TN091,5.0,CaE,526515,964.949459,48297.57,"MULTIPOLYGON (((1242604.1 1578393.6, 1242606.3...",CaE,"Calvin channery silt loam, 20 to 35 percent sl...",None,...,Very limited,Very limited,Poor,Poor,Very limited,1.0,Erosion hazard moderate,0,1.000,526515


In [17]:
soil_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1315000 entries, 0 to 1314999
Data columns (total 47 columns):
 #   Column          Non-Null Count    Dtype   
---  ------          --------------    -----   
 0   AREASYMBOL      1315000 non-null  object  
 1   SPATIALVER      1315000 non-null  float64 
 2   MUSYM           1315000 non-null  object  
 3   MUKEY           1315000 non-null  object  
 4   Shape_Length    1315000 non-null  float64 
 5   Shape_Area      1315000 non-null  float64 
 6   geometry        1315000 non-null  geometry
 7   musym           1315000 non-null  object  
 8   muname          1315000 non-null  object  
 9   mustatus        0 non-null        object  
 10  slopegraddcp    1273270 non-null  float32 
 11  slopegradwta    1274060 non-null  float32 
 12  brockdepmin     362838 non-null   float64 
 13  wtdepannmin     373441 non-null   float64 
 14  wtdepaprjunmin  253357 non-null   float64 
 15  flodfreqdcd     1267934 non-null  object  
 16  flodfreqma

Keep only required soil features like water availability features("aws025wta","aws050wta","aws0100wta","aws0150wta"), Drainage features(hydgrpdcd,drclassdcd,drclasswettest,hydclprs), Fllod risk ("flodfreqdcd","flodfreqmax"), Terrain(slopegraddcp,slopegradwta), Soil Depth (brockdepmin,wtdepannmin,wtdepaprjunmin)

In [18]:
soil_gdf_prop = soil_gdf[["geometry","aws025wta","aws050wta","aws0100wta","aws0150wta","hydgrpdcd","drclassdcd","drclasswettest","slopegraddcp","slopegradwta","brockdepmin","wtdepannmin","wtdepaprjunmin"]]

In [19]:
tn_counties

,county,geometry
29,Hamilton,"POLYGON ((-85.17523 34.98598, -85.17663 34.986..."
64,Marion,"POLYGON ((-85.86396 34.99316, -85.86395 34.993..."
67,White,"MULTIPOLYGON (((-85.6446 36.01505, -85.64392 3..."
127,Morgan,"POLYGON ((-84.45834 36.18654, -84.45834 36.186..."
180,Campbell,"POLYGON ((-84.09196 36.22261, -84.09202 36.222..."
...,...,...
3080,Decatur,"POLYGON ((-87.97643 35.50673, -87.9767 35.5058..."
3085,Maury,"POLYGON ((-86.82054 35.63207, -86.8207 35.6308..."
3103,Fayette,"POLYGON ((-89.3388 35.40016, -89.33509 35.4000..."
3108,Hamblen,"POLYGON ((-83.24943 36.28716, -83.24935 36.287..."


#### Calculate the percentage of county for soil type

In [20]:
# Convert soil_gdf_prop and tn_counties to equal area projection(epsg=5070)
soil_gdf_prop = soil_gdf.to_crs(epsg=5070)
tn_counties = tn_counties.to_crs(epsg=5070)

In [21]:
#Compute overlapping Geometries
intersections = gpd.overlay(soil_gdf_prop, tn_counties, how='intersection')


In [23]:
intersections

,AREASYMBOL,SPATIALVER,MUSYM,MUKEY,Shape_Length,Shape_Area,musym,muname,mustatus,slopegraddcp,...,engcmssdcd,engcmssmp,urbrecptdcd,urbrecptwta,forpehrtdcp,hydclprs,awmmfpwwta,mukey,county,geometry
0,TN091,5.0,CaF,526516,5236.926602,534416.620,CaF,"Calvin channery silt loam, 35 to 50 percent sl...",None,43.0,...,Poor,Poor,Very limited,1.000,Erosion hazard severe,0,1.000,526516,Johnson,"POLYGON ((1259510.8 1600181.8, 1259501.3 16002..."
1,TN091,5.0,CcE,526518,2847.382752,223007.820,CcE,"Cataska channery silt loam, 20 to 35 percent s...",None,28.0,...,Poor,Fair,Very limited,1.000,Erosion hazard moderate,0,1.000,526518,Johnson,"POLYGON ((1244679.2 1577094.1, 1244664.4 15771..."
2,TN091,5.0,DtE,526529,2299.663378,79289.810,DtE,"Ditney sandy loam, 20 to 35 percent slopes",None,28.0,...,Fair,Fair,Very limited,1.000,Erosion hazard moderate,0,1.000,526529,Johnson,"POLYGON ((1259693 1590991.9, 1259679.1 1590984..."
3,TN091,5.0,KeC,526541,1132.074101,77883.950,KeC,"Keener loam, 5 to 12 percent slopes",None,9.0,...,Poor,Poor,Very limited,1.000,Erosion hazard severe,0,0.622,526541,Johnson,"POLYGON ((1258526.7 1590355.8, 1258521 1590354..."
4,TN091,5.0,CaE,526515,964.949459,48297.570,CaE,"Calvin channery silt loam, 20 to 35 percent sl...",None,28.0,...,Poor,Poor,Very limited,1.000,Erosion hazard moderate,0,1.000,526515,Johnson,"POLYGON ((1242606.3 1578410.7, 1242636.5 15784..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1327442,TN157,11.0,W,567324,104.434694,770.015,W,Water,None,NaN,...,Not rated,Not rated,Not rated,NaN,Not rated,0,NaN,567324,Shelby,"POLYGON ((541530.9 1387736.2, 541522.7 1387745..."
1327443,TN157,11.0,GaB2,567294,1841.788500,53532.270,GaB2,"Grenada silt loam, 2 to 5 percent slopes, mode...",None,4.0,...,Poor,Poor,Somewhat limited,0.082,Erosion hazard moderate,0,1.000,567294,Shelby,"POLYGON ((551483.1 1387911.2, 551488.4 1387805..."
1327444,TN157,11.0,Ad,3116427,13177.131883,554053.995,Ad,"Adler silt loam, 0 to 2 percent slopes, occasi...",None,1.0,...,Poor,Poor,Somewhat limited,0.062,Erosion hazard slight,0,0.999,3116427,Shelby,"POLYGON ((552519.3 1388245.3, 552491.7 1388249..."
1327445,TN157,11.0,Ad,3116427,13177.131883,554053.995,Ad,"Adler silt loam, 0 to 2 percent slopes, occasi...",None,1.0,...,Poor,Poor,Somewhat limited,0.062,Erosion hazard slight,0,0.999,3116427,Tipton,"MULTIPOLYGON (((551800.4 1388373.2, 551833 138..."


In [24]:
#Area of Soil Polygon piece
intersections["soil_area"]= intersections.geometry.area

In [25]:
# Area of County polygon
county_area = tn_counties.copy()
county_area["county_area"] = county_area.geometry.area

In [26]:
# Merge two data frames
soil_county_intersection = intersections.merge(county_area, on="county")

In [27]:
# Calculate the percentage of county for soil type
soil_county_intersection["percent"] = soil_county_intersection["soil_area"] / soil_county_intersection["county_area"]

In [28]:
soil_county_intersection

,AREASYMBOL,SPATIALVER,MUSYM,MUKEY,Shape_Length,Shape_Area,musym,muname,mustatus,slopegraddcp,...,forpehrtdcp,hydclprs,awmmfpwwta,mukey,county,geometry_x,soil_area,geometry_y,county_area,percent
0,TN091,5.0,CaF,526516,5236.926602,534416.620,CaF,"Calvin channery silt loam, 35 to 50 percent sl...",None,43.0,...,Erosion hazard severe,0,1.000,526516,Johnson,"POLYGON ((1259510.8 1600181.8, 1259501.3 16002...",534416.620000,"POLYGON ((1263802.12 1584677.439, 1263798.528 ...",7.839640e+08,6.816851e-04
1,TN091,5.0,CcE,526518,2847.382752,223007.820,CcE,"Cataska channery silt loam, 20 to 35 percent s...",None,28.0,...,Erosion hazard moderate,0,1.000,526518,Johnson,"POLYGON ((1244679.2 1577094.1, 1244664.4 15771...",223007.820000,"POLYGON ((1263802.12 1584677.439, 1263798.528 ...",7.839640e+08,2.844618e-04
2,TN091,5.0,DtE,526529,2299.663378,79289.810,DtE,"Ditney sandy loam, 20 to 35 percent slopes",None,28.0,...,Erosion hazard moderate,0,1.000,526529,Johnson,"POLYGON ((1259693 1590991.9, 1259679.1 1590984...",79289.810000,"POLYGON ((1263802.12 1584677.439, 1263798.528 ...",7.839640e+08,1.011396e-04
3,TN091,5.0,KeC,526541,1132.074101,77883.950,KeC,"Keener loam, 5 to 12 percent slopes",None,9.0,...,Erosion hazard severe,0,0.622,526541,Johnson,"POLYGON ((1258526.7 1590355.8, 1258521 1590354...",77883.950000,"POLYGON ((1263802.12 1584677.439, 1263798.528 ...",7.839640e+08,9.934633e-05
4,TN091,5.0,CaE,526515,964.949459,48297.570,CaE,"Calvin channery silt loam, 20 to 35 percent sl...",None,28.0,...,Erosion hazard moderate,0,1.000,526515,Johnson,"POLYGON ((1242606.3 1578410.7, 1242636.5 15784...",48297.570000,"POLYGON ((1263802.12 1584677.439, 1263798.528 ...",7.839640e+08,6.160687e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1327442,TN157,11.0,W,567324,104.434694,770.015,W,Water,None,NaN,...,Not rated,0,NaN,567324,Shelby,"POLYGON ((541530.9 1387736.2, 541522.7 1387745...",770.015000,"POLYGON ((573880.443 1369498.055, 573882.017 1...",2.023033e+09,3.806240e-07
1327443,TN157,11.0,GaB2,567294,1841.788500,53532.270,GaB2,"Grenada silt loam, 2 to 5 percent slopes, mode...",None,4.0,...,Erosion hazard moderate,0,1.000,567294,Shelby,"POLYGON ((551483.1 1387911.2, 551488.4 1387805...",53532.270000,"POLYGON ((573880.443 1369498.055, 573882.017 1...",2.023033e+09,2.646139e-05
1327444,TN157,11.0,Ad,3116427,13177.131883,554053.995,Ad,"Adler silt loam, 0 to 2 percent slopes, occasi...",None,1.0,...,Erosion hazard slight,0,0.999,3116427,Shelby,"POLYGON ((552519.3 1388245.3, 552491.7 1388249...",507721.435671,"POLYGON ((573880.443 1369498.055, 573882.017 1...",2.023033e+09,2.509704e-04
1327445,TN157,11.0,Ad,3116427,13177.131883,554053.995,Ad,"Adler silt loam, 0 to 2 percent slopes, occasi...",None,1.0,...,Erosion hazard slight,0,0.999,3116427,Tipton,"MULTIPOLYGON (((551800.4 1388373.2, 551833 138...",46332.559329,"MULTIPOLYGON (((523919.882 1386057.923, 524093...",1.212730e+09,3.820516e-05


In [75]:
soil_county_intersection.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1327447 entries, 0 to 1327446
Data columns (total 61 columns):
 #   Column                   Non-Null Count    Dtype   
---  ------                   --------------    -----   
 0   AREASYMBOL               1327447 non-null  object  
 1   SPATIALVER               1327447 non-null  float64 
 2   MUSYM                    1327447 non-null  object  
 3   MUKEY                    1327447 non-null  object  
 4   Shape_Length             1327447 non-null  float64 
 5   Shape_Area               1327447 non-null  float64 
 6   musym                    1327447 non-null  object  
 7   muname                   1327447 non-null  object  
 8   mustatus                 0 non-null        object  
 9   slopegraddcp             1285203 non-null  float32 
 10  slopegradwta             1286013 non-null  float32 
 11  brockdepmin              361453 non-null   float64 
 12  wtdepannmin              380599 non-null   float64 
 13  wtdepaprjunmin     

In [29]:
# Convert Soil map units into county level labeled soil data
# soil_with_county = gpd.sjoin(
#     soil_gdf_prop,
#     tn_counties,
#     how="left",
#     predicate="intersects" #match if geometris touch or overlap
# )

In [30]:
soil_county_intersection.shape

(1327447, 52)

In [44]:
# Soil Numeric columns
num_cols = ["aws025wta","aws050wta","aws0100wta","aws0150wta",
            "slopegraddcp","slopegradwta",
            "brockdepmin","wtdepannmin","wtdepaprjunmin"
           ]


In [45]:
#Aggregate numerical columns by using weighted mean(since it has area percentage using weighted mean)
for col in num_cols:
    soil_county_intersection[f"{col}_weighted"] = (soil_county_intersection[col]* soil_county_intersection["percent"])

In [46]:
 soil_county_intersection["aws025wta_weighted"]

0          0.002181
1          0.000842
2          0.000304
3          0.000377
4          0.000197
             ...   
1327442         NaN
1327443    0.000147
1327444    0.001410
1327445    0.000215
1327446    0.000091
Name: aws025wta_weighted, Length: 1327447, dtype: float64

In [51]:
weighted_cols = [f"{col}_weighted" for col in num_cols]
soil_num_weighted_mean = soil_county_intersection.groupby("county")[weighted_cols].sum().reset_index()

In [52]:
#Rename weighted columns for consistency
soil_num_weighted_mean = soil_num_weighted_mean.rename(columns={
    "aws025wta_weighted": "aws025wta",
    "aws050wta_weighted": "aws050wta",
    "aws0100wta_weighted": "aws0100wta",
    "aws0150wta_weighted": "aws0150wta",
    "slopegraddcp_weighted": "slopegraddcp",
    "slopegradwta_weighted": "slopegradwta",
    "brockdepmin_weighted": "brockdepmin",
    "wtdepannmin_weighted": "wtdepannmin",
    "wtdepaprjunmin_weighted": "wtdepaprjunmin"
})

In [53]:
soil_num_weighted_mean.head()

,county,aws025wta,aws050wta,aws0100wta,aws0150wta,slopegraddcp,slopegradwta,brockdepmin,wtdepannmin,wtdepaprjunmin
0,Anderson,3.156171,5.772640,9.153774,11.978131,28.770325,28.367397,23.298704,7.506619,2.767103
1,Bedford,4.025817,7.452896,13.134498,16.623633,9.777086,9.769806,60.060881,15.654474,3.823853
2,Benton,3.828079,7.269015,12.165571,16.159500,12.640946,12.038437,32.082159,17.723619,17.107790
3,Bledsoe,3.599513,6.704664,10.377442,12.448807,18.632131,18.588435,47.961418,3.658212,0.341449
4,Blount,3.506793,6.482429,10.747538,13.605522,27.598879,27.577185,38.113855,11.118141,1.426487


In [54]:
#Aggregate numerical columns
#soil_num = soil_with_county.groupby("county")[num_cols].mean().reset_index()

In [55]:
cat_cols = ["hydgrpdcd","drclassdcd","drclasswettest"]

In [56]:
#aggregation of categorical column
#soil_county_intersection.groupby(["county","hydgrpdcd"])["percent"].sum()

In [66]:
#Aggregation of Categorical Columns

soil_cat_pct = []
for col in cat_cols:
    temp = soil_county_intersection.groupby(["county", col])["percent"].sum().reset_index()
    temp = temp.pivot_table(
        index="county",
        columns=col,
        values="percent",
        fill_value=0
    )

    # adding the category name in front of column names to keep them unique during merge
    temp = temp.add_prefix(f"{col}_")
    temp = temp.reset_index()
    soil_cat_pct.append(temp)

In [67]:
#Convert soil_cat_pct list in to date frame
#convet each list into df and merge them
soil_cat_pct_df = soil_cat_pct[0]

for df in soil_cat_pct[1:]:
    soil_cat_pct_df = soil_cat_pct_df.merge(df, on="county", how="outer")

In [68]:
soil_cat_pct_df.columns

Index(['county', 'hydgrpdcd_A', 'hydgrpdcd_B', 'hydgrpdcd_B/D', 'hydgrpdcd_C',
       'hydgrpdcd_C/D', 'hydgrpdcd_D', 'drclassdcd_Excessively drained',
       'drclassdcd_Moderately well drained', 'drclassdcd_Poorly drained',
       'drclassdcd_Somewhat excessively drained',
       'drclassdcd_Somewhat poorly drained', 'drclassdcd_Very poorly drained',
       'drclassdcd_Well drained', 'drclasswettest_Excessively drained',
       'drclasswettest_Moderately well drained',
       'drclasswettest_Poorly drained',
       'drclasswettest_Somewhat excessively drained',
       'drclasswettest_Somewhat poorly drained',
       'drclasswettest_Very poorly drained', 'drclasswettest_Well drained'],
      dtype='object')

In [77]:
# Merge Numerical and categorical Data 
soil_prop_data = soil_num_weighted_mean.merge(
    soil_cat_pct_df,
    on="county",
    how="left"
)

In [80]:
#Export  Soil Properties Data frame to csv file
soil_prop_data.to_csv('../data/soil_prop_data _df', index=False)

In [78]:
soil_prop_data.head()

,county,aws025wta,aws050wta,aws0100wta,aws0150wta,slopegraddcp,slopegradwta,brockdepmin,wtdepannmin,wtdepaprjunmin,...,drclassdcd_Somewhat poorly drained,drclassdcd_Very poorly drained,drclassdcd_Well drained,drclasswettest_Excessively drained,drclasswettest_Moderately well drained,drclasswettest_Poorly drained,drclasswettest_Somewhat excessively drained,drclasswettest_Somewhat poorly drained,drclasswettest_Very poorly drained,drclasswettest_Well drained
0,Anderson,3.156171,5.772640,9.153774,11.978131,28.770325,28.367397,23.298704,7.506619,2.767103,...,0.009593,0.000000,0.875085,3.461202e-07,0.069925,0.000525,0.016732,0.009593,0.000000,0.869346
1,Bedford,4.025817,7.452896,13.134498,16.623633,9.777086,9.769806,60.060881,15.654474,3.823853,...,0.056672,0.000000,0.751918,0.000000e+00,0.117521,0.024722,0.043160,0.056672,0.000000,0.751933
2,Benton,3.828079,7.269015,12.165571,16.159500,12.640946,12.038437,32.082159,17.723619,17.107790,...,0.053539,0.002896,0.426720,3.175618e-03,0.384051,0.017688,0.024201,0.042626,0.002896,0.428553
3,Bledsoe,3.599513,6.704664,10.377442,12.448807,18.632131,18.588435,47.961418,3.658212,0.341449,...,0.003014,0.000000,0.776495,0.000000e+00,0.022260,0.016520,0.180254,0.003014,0.000000,0.776545
4,Blount,3.506793,6.482429,10.747538,13.605522,27.598879,27.577185,38.113855,11.118141,1.426487,...,0.000000,0.000368,0.705347,5.161192e-02,0.044212,0.013774,0.141022,0.003621,0.000368,0.703523


In [79]:
soil_prop_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 30 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   county                                       95 non-null     object 
 1   aws025wta                                    95 non-null     float64
 2   aws050wta                                    95 non-null     float64
 3   aws0100wta                                   95 non-null     float64
 4   aws0150wta                                   95 non-null     float64
 5   slopegraddcp                                 95 non-null     float64
 6   slopegradwta                                 95 non-null     float64
 7   brockdepmin                                  95 non-null     float64
 8   wtdepannmin                                  95 non-null     float64
 9   wtdepaprjunmin                               95 non-null     float64
 10  hydg

##### Gather PH values of soil 

In [76]:
comp = gpd.read_file("../data/gSSURGO_TN.gdb", layer="component")

In [81]:
comp.head()

,comppct_l,comppct_r,comppct_h,compname,compkind,majcompflag,otherph,localphase,slope_l,slope_r,...,flsoilleachpot,flsoirunoffpot,fltemik2use,fltriumph2use,indraingrp,innitrateleachi,misoimgmtgrp,vasoimgtgrp,mukey,cokey
0,NaN,10,NaN,Minor components,None,No,None,None,NaN,NaN,...,None,None,N/A,N/A,None,NaN,None,None,526504,26597851
1,NaN,40,NaN,Ashe,Series,Yes,None,None,50.0,73.0,...,None,None,N/A,N/A,None,NaN,None,None,526504,26597852
2,NaN,30,NaN,Cleveland,Series,Yes,None,None,50.0,73.0,...,None,None,N/A,N/A,None,NaN,None,None,526504,26597853
3,NaN,20,NaN,Rock outcrop,Miscellaneous area,Yes,None,None,50.0,73.0,...,None,None,N/A,N/A,None,NaN,None,None,526504,26597854
4,NaN,100,NaN,Ashe,Series,Yes,None,None,12.0,19.0,...,None,None,N/A,N/A,None,NaN,None,None,526505,26597794


In [82]:
comp.columns

Index(['comppct_l', 'comppct_r', 'comppct_h', 'compname', 'compkind',
       'majcompflag', 'otherph', 'localphase', 'slope_l', 'slope_r',
       ...
       'flsoilleachpot', 'flsoirunoffpot', 'fltemik2use', 'fltriumph2use',
       'indraingrp', 'innitrateleachi', 'misoimgmtgrp', 'vasoimgtgrp', 'mukey',
       'cokey'],
      dtype='object', length=109)

In [85]:
comp.columns.tolist()

['comppct_l',
 'comppct_r',
 'comppct_h',
 'compname',
 'compkind',
 'majcompflag',
 'otherph',
 'localphase',
 'slope_l',
 'slope_r',
 'slope_h',
 'slopelenusle_l',
 'slopelenusle_r',
 'slopelenusle_h',
 'runoff',
 'tfact',
 'wei',
 'weg',
 'erocl',
 'earthcovkind1',
 'earthcovkind2',
 'hydricon',
 'hydricrating',
 'drainagecl',
 'elev_l',
 'elev_r',
 'elev_h',
 'aspectccwise',
 'aspectrep',
 'aspectcwise',
 'geomdesc',
 'albedodry_l',
 'albedodry_r',
 'albedodry_h',
 'airtempa_l',
 'airtempa_r',
 'airtempa_h',
 'map_l',
 'map_r',
 'map_h',
 'reannualprecip_l',
 'reannualprecip_r',
 'reannualprecip_h',
 'ffd_l',
 'ffd_r',
 'ffd_h',
 'nirrcapcl',
 'nirrcapscl',
 'nirrcapunit',
 'irrcapcl',
 'irrcapscl',
 'irrcapunit',
 'cropprodindex',
 'constreeshrubgrp',
 'wndbrksuitgrp',
 'rsprod_l',
 'rsprod_r',
 'rsprod_h',
 'foragesuitgrpid',
 'wlgrain',
 'wlgrass',
 'wlherbaceous',
 'wlshrub',
 'wlconiferous',
 'wlhardwood',
 'wlwetplant',
 'wlshallowwat',
 'wlrangeland',
 'wlopenland',
 'wlwood

PH Values are not in the Copmponent Table. Check chorizon table

In [87]:
chorizon = gpd.read_file("../data/gSSURGO_TN.gdb", layer="chorizon")

In [88]:
chorizon.columns.tolist()

['hzname',
 'desgndisc',
 'desgnmaster',
 'desgnmasterprime',
 'desgnvert',
 'hzdept_l',
 'hzdept_r',
 'hzdept_h',
 'hzdepb_l',
 'hzdepb_r',
 'hzdepb_h',
 'hzthk_l',
 'hzthk_r',
 'hzthk_h',
 'fraggt10_l',
 'fraggt10_r',
 'fraggt10_h',
 'frag3to10_l',
 'frag3to10_r',
 'frag3to10_h',
 'sieveno4_l',
 'sieveno4_r',
 'sieveno4_h',
 'sieveno10_l',
 'sieveno10_r',
 'sieveno10_h',
 'sieveno40_l',
 'sieveno40_r',
 'sieveno40_h',
 'sieveno200_l',
 'sieveno200_r',
 'sieveno200_h',
 'sandtotal_l',
 'sandtotal_r',
 'sandtotal_h',
 'sandvc_l',
 'sandvc_r',
 'sandvc_h',
 'sandco_l',
 'sandco_r',
 'sandco_h',
 'sandmed_l',
 'sandmed_r',
 'sandmed_h',
 'sandfine_l',
 'sandfine_r',
 'sandfine_h',
 'sandvf_l',
 'sandvf_r',
 'sandvf_h',
 'silttotal_l',
 'silttotal_r',
 'silttotal_h',
 'siltco_l',
 'siltco_r',
 'siltco_h',
 'siltfine_l',
 'siltfine_r',
 'siltfine_h',
 'claytotal_l',
 'claytotal_r',
 'claytotal_h',
 'claysizedcarb_l',
 'claysizedcarb_r',
 'claysizedcarb_h',
 'om_l',
 'om_r',
 'om_h',
 'dbte

In [89]:
[col for col in chorizon.columns if "ph" in col.lower()]

['ph1to1h2o_l',
 'ph1to1h2o_r',
 'ph1to1h2o_h',
 'ph01mcacl2_l',
 'ph01mcacl2_r',
 'ph01mcacl2_h',
 'ph2osoluble_l',
 'ph2osoluble_r',
 'ph2osoluble_h']